In this notebook I will clean and prepare a movie dataset for TF-IDF vectorization, the vectors from which will be used for a content-based movie-recommendation system.

Data retrieved from: https://www.kaggle.com/datasets/ggtejas/tmdb-imdb-merged-movies-dataset

In [ ]:
#imports
import pandas as pd
from scipy.sparse import save_npz
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
#mount drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def prep_f(f):
  if pd.isnull(f):
    return []
  return [term.strip().lower().replace(" ","-") for term in f.split(",")]         #join multiple-word items with hyphens

def concat_row(row):
  return " ".join(row['genres']+ row['keywords']+row['directors']+row['cast'])    #convert rows data into one string



In [ ]:
#obtain movie dataset
path = #path for dataset
movies_df=pd.read_csv(path + "/movie_set.csv")

#clean dataset
movies_df.dropna()                                                              #drop null values
movies_df_cleaned=movies_df[
    (movies_df['vote_count']>500) &
    (movies_df['budget']>1000000)
]
movies_df_cleaned['release_date']=pd.to_datetime(movies_df_cleaned['release_date'])
movies_df_cleaned['release_year']=movies_df_cleaned['release_date'].dt.year
features=['genres','keywords','directors','cast']                               #select relevant columns
movies_df_cleaned=movies_df_cleaned[features + ['title']+['release_year']]

#create concat feature
for feature in features:
  movies_df_cleaned[feature]=movies_df_cleaned[feature].apply(prep_f)
movies_df_cleaned['concat']=movies_df_cleaned.apply(concat_row,axis=1)



In [ ]:
tf=TfidfVectorizer(
    stop_words='english',
    token_pattern=r'(?u)[\w-]+')                                                #preserve hyphenated words
tf_matrix=tf.fit_transform(movies_df_cleaned['concat'])                         #create vector for each movie

In [ ]:
#export movie titles
movies_export=movies_df_cleaned[['title','release_year','concat']]
movies_export.to_csv(path + "/movies_db.csv",index=False)
#export sparse matrix
save_npz(path + "/tf_matrix.npz",tf_matrix)